In [ ]:
import pandas as pd
import json
import numpy as np
import re
import os

import matplotlib.pyplot as plt 
import seaborn as sns
import matplotlib.patches as mpatches
from IPython.core.pylabtools import figsize
from matplotlib_inline.backend_inline import set_matplotlib_formats
from statannotations.Annotator import Annotator
from matplotlib_venn import venn3_unweighted
import matplotlib.cm as cm
#import tikzplotlib 
import matplotlib.colors as mcolors
from tensorboard.backend.event_processing import event_accumulator
import tensorflow as tf

def blend_colors(hex_list):
    """Blend multiple hex colors by averaging their RGB values."""
    rgbs = [mcolors.to_rgb(c) for c in hex_list]
    blended = tuple(sum(vals)/len(vals) for vals in zip(*rgbs))
    return mcolors.to_hex(blended)

%matplotlib inline
set_matplotlib_formats('retina') # high-res display in notebook
# Set the Seaborn theme
context = 'paper'  # paper, talk, or notebook

save_settings = {'format': 'svg'}  # always use svg as it allows scaling figures up & down without losing resolution! 

PALETTE = {
    "SAD (ours)": "#115e17",
    "solver-aware": "#115e17",
    "SAD Oracle": "#538d58",
    "$\mathcal{L}_{\\text{sup}}$ ablation": "#6e796c",
    "Solver-blind baseline": "#6e796c",
    "solver-blind": "#6e796c",
    "Solver-blind Oracle": "#b0b9af",
    "Exedec": "#2F4F4F",
    "GT Oracle": "#445469"
}

if context == 'talk' or context == 'paper':
    sns.set_theme(context=context, style='whitegrid', palette=sns.color_palette(PALETTE.values()), font_scale=1.75)
else:
    sns.set_theme(context=context, style='whitegrid', palette=sns.color_palette(PALETTE.values()), font_scale=1.2)

In [ ]:
# Where the run_*.sh scripts wrote their results (RESULTS_DIR), relative to plotting/.
RESULTS_DIR = '../results'
EVAL_DIR = f'{RESULTS_DIR}/evaluation'
os.makedirs('../plots/svg', exist_ok=True)

SEEDS = [10, 20, 30, 40, 50]

SYNTH_MODEL = "synthesizer_model"
EXPERIMENTS = ["NONE", "LENGTH_GENERALIZATION"]
EXP_LBL_MAP = {"NONE": 'Test on training\ndistribution', "LENGTH_GENERALIZATION": 'Length generalization'}
PT_MAPPING = {"supervised": "Solver-blind baseline", "exedec": "Exedec", "baseline": "No-Decomposition Ablation", 'grpo': 'SAD (ours)', 'grpo_kbest': 'SAD Oracle', 'supervised_kbest': 'Solver-blind Oracle', 'supervised_oracle': 'GT Oracle'}
HUE_ORDER = [
    "SAD (ours)", 
    "SAD Oracle", 
    "Solver-blind baseline", 
    "Solver-blind Oracle", 
    "Exedec", 
    "GT Oracle"
]

### Method Definition

In [ ]:
def load_accuracy_data(ds, path, beam_size=10):
    accuracies = {"SAD (ours)": {},  "SAD Oracle": {}, "Solver-blind baseline": {}, "Solver-blind Oracle": {}, "Exedec": {}, "No-Decomposition Ablation": {}, "GT Oracle": {}}

    for training_type in ["supervised", 'exedec', "grpo", "grpo_kbest", "supervised_kbest", "supervised_oracle"]:
        gr, ca, synth = True, False, ',synthesizer=nsa'

        if training_type == 'baseline':
            prediction_type = 'baseline'
            tt = 'supervised'
            if ds != 'lambdabeam':
                gr, ca = False, True
        elif training_type == 'exedec':
            prediction_type = 'separate'
            tt = 'exedec'
            gr, ca, synth = False, True, ''
        else:
            prediction_type = 'separate'
            tt = training_type
        oracle_type = ''
        if '_' not in training_type:
            oracle_type = '_none'

        p = path + f"/{tt}{oracle_type}_nsa{'false' if tt == 'exedec' else 'true'}/end_to_end_predict-{ds}-run-e2e_predict_1-"
        
            
        exp = {k: [np.nan] * 5 for k in EXPERIMENTS}
        for experiment in EXPERIMENTS:
            for j, seed in enumerate(SEEDS):
                
                try:
                    with open(p + prediction_type  + f'/tb/hparams-dataset_type={ds},prediction_type={prediction_type},experiment={experiment},beam_size={beam_size},seed={seed},greedy={gr},compute_align={ca}{synth}/results-{prediction_type}.json') as f:
                        data = json.load(f)
                except FileNotFoundError:
                    print(p + prediction_type  + f'/tb/hparams-dataset_type={ds},prediction_type={prediction_type},experiment={experiment},beam_size={beam_size},seed={seed},greedy={gr},compute_align={ca}{synth}/results-{prediction_type}.json')
                    continue

                num_solved = len([ele for ele in data if ele["success"]])
                true_positives = sum(x['false_positive'] is False for x in data)
                exp[experiment][j] = (true_positives) / len(data) * 100 if len(data) > 0  else np.nan
        accuracies[PT_MAPPING[training_type]] = exp
    return accuracies


def load_accuracy_by_length(ds, path, beam_size=10):
    data_by_type = {"SAD (ours)": [], "Exedec": [], "Solver-blind baseline": []}

    for training_type in ['grpo', 'exedec', "supervised"]:
        gr, ca, synth = True, False, ',synthesizer=nsa'

        if training_type == 'baseline':
            prediction_type = 'baseline'
            tt = 'supervised'
            if ds != 'lambdabeam':
                gr, ca = False, True
        elif training_type == 'exedec':
            prediction_type = 'separate'
            tt = 'exedec'
            gr, ca, synth = False, True, ''
        else:
            prediction_type = 'separate'
            tt = training_type
        oracle_type = ''
        if '_' not in training_type:
            oracle_type = '_none'

        p = path + f"/{tt}{oracle_type}_nsa{'false' if tt == 'exedec' else 'true'}/end_to_end_predict-{ds}-run-e2e_predict_1-"
        
            
        exp = {k: [np.nan] * 5 for k in EXPERIMENTS}
        for experiment in EXPERIMENTS:
            for j, seed in enumerate(SEEDS):
                
                try:
                    with open(p + prediction_type  + f'/tb/hparams-dataset_type={ds},prediction_type={prediction_type},experiment={experiment},beam_size={beam_size},seed={seed},greedy={gr},compute_align={ca}{synth}/results-{prediction_type}.json') as f:
                        data = json.load(f)
                except FileNotFoundError:
                    print(p + prediction_type  + f'/tb/hparams-dataset_type={ds},prediction_type={prediction_type},experiment={experiment},beam_size={beam_size},seed={seed},greedy={gr},compute_align={ca}{synth}/results-{prediction_type}.json')
                    continue
                for ele in data:
                    data_by_type[PT_MAPPING[training_type]].append({
                        "length": ele["ground_truth_length"],
                        "success": ele["false_positive"] is False,
                        'experiment': experiment,
                        'seed': seed
                    })

    return data_by_type


def load_solutions_and_gt(ds, path, beam_size=10):
    data_by_type = {"SAD (ours)": [], "Exedec": [], "Solver-blind baseline": []}

    for training_type in ['grpo', 'exedec', "supervised"]:
        gr, ca, synth = True, False, ',synthesizer=nsa'

        if training_type == 'baseline':
            prediction_type = 'baseline'
            tt = 'supervised'
            if ds != 'lambdabeam':
                gr, ca = False, True
        elif training_type == 'exedec':
            prediction_type = 'separate'
            tt = 'exedec'
            gr, ca, synth = False, True, ''
        else:
            prediction_type = 'separate'
            tt = training_type
        oracle_type = ''
        if '_' not in training_type:
            oracle_type = '_none'

        p = path + f"/{tt}{oracle_type}_nsa{'false' if tt == 'exedec' else 'true'}/end_to_end_predict-{ds}-run-e2e_predict_1-"
        if ds == 'lambdabeam' and training_type == 'exedec':
            p = path + f"/supervised{oracle_type}_nsafalse/end_to_end_predict-{ds}-run-e2e_predict_1-"
            gr, ca, synth = True, False, ''
        
            
        exp = {k: [np.nan] * 5 for k in EXPERIMENTS}
        for experiment in EXPERIMENTS:
            for j, seed in enumerate(SEEDS):
                
                try:
                    with open(p + prediction_type  + f'/tb/hparams-dataset_type={ds},prediction_type={prediction_type},experiment={experiment},beam_size={beam_size},seed={seed},greedy={gr},compute_align={ca}{synth}/results-{prediction_type}.json') as f:
                        data = json.load(f)
                except FileNotFoundError:
                    print(p + prediction_type  + f'/tb/hparams-dataset_type={ds},prediction_type={prediction_type},experiment={experiment},beam_size={beam_size},seed={seed},greedy={gr},compute_align={ca}{synth}/results-{prediction_type}.json')
                    continue
                for ele in data:
                    if ele["false_positive"] is False:
                        data_by_type[PT_MAPPING[training_type]].append({
                            "solution": ele['solution'],
                            "gt": ele['ground_truth'],
                            'seed': seed,
                            'success': True,
                            'category': experiment,
                            'task_idx': ele['test_example_index']
                        })

    return data_by_type


In [ ]:


metric = 'test/beam-size-10' # synthesizer/test/beam-size-10 for synthesizer single step acc
rows = []
dataset = 'deepcoder'

for approach in ['grpo', 'supervised']:
    log_dir = f"{RESULTS_DIR}/exedec_train_{dataset}_run-1/spec_decomposer_model/{approach}/tb/"

    for experiment in EXPERIMENTS:
        for j, seed in enumerate(SEEDS):
            if dataset == 'robustfill':
                ev_dir = log_dir + f'adr=0.1,ara=True,dr=0.1,e={experiment},ed=512,hd=1024,l=0.0001,md=200,mpced=800,npb=32,s={seed},scnpr=0.0,ura=True,dm=coupled/'
            else:
                ev_dir = log_dir + f'adr=0.1,ara=True,dr=0.1,e={experiment},ed=512,hd=1024,l=0.0001,md=60,mpced=180,npb=32,s={seed},scnpr=0.0,ura=True,dm=coupled/'
            try:
                ea = event_accumulator.EventAccumulator(ev_dir)
                ea.Reload()
                events = ea.Tensors(metric)
            except:
                continue

            if not events:
                continue

            last_event = events[-1]
            value = tf.make_ndarray(last_event.tensor_proto).item()

            rows.append({
                "seed": seed,
                "approach": 'solver-aware' if approach == 'grpo' else 'solver-blind' ,#PT_MAPPING[approach],   # or parse from path if needed
                "experiment": EXP_LBL_MAP[experiment],
                "accuracy": value,
                "step": ea.Tensors(metric)[-1].step
            })

# Create DataFrame
df = pd.DataFrame(rows)

df = df[df['step'] == 499999]
df = df[df['accuracy'] != 0]

stats = df.groupby(['approach', 'experiment'])['accuracy'].agg(['mean', 'std', 'count']).reset_index()
stats['sem'] = stats['std'] / np.sqrt(stats['count'])

fig, ax = plt.subplots()
sns.barplot(data=df, x='experiment', y='accuracy', hue='approach', errorbar=('ci', 95), ax=ax, err_kws={'color': 'gray'}, palette=PALETTE)

# Compute pairwise comparisons between all approaches per category
hue_types = df['approach'].unique()
x_categories = df['experiment'].unique()

# Create list of all pairs for comparisons
comparisons = []
for cat in x_categories:
    cat_pairs = [(h1, h2) for i, h1 in enumerate(hue_types) for j, h2 in enumerate(hue_types) if i < j]
    for pair in cat_pairs:
        comparisons.append((pair, cat))

# Restructure comparisons for statannotations
formatted_comparisons = []
for (h1, h2), cat in comparisons:
    formatted_comparisons.append(((cat, h1), (cat, h2)))

# Create annotator
"""annotator = Annotator(ax, formatted_comparisons, data=df_plot, x='experiment', y='accuracy', hue='approach', order=list(x_categories))
annotator.configure(test='t-test_paired', text_format='star', loc='outside', verbose=2)
annotator.apply_test()
annotator.annotate(
    line_offset=0.1,           
)"""
plt.gcf().set_figwidth(7)

# Add SEM error bars manually
hue_types = df['approach'].unique()
num_hues = len(hue_types)
x_categories = list(df['experiment'].unique())
width = 0.8 / num_hues
offsets = np.linspace(-0.4 + width/2, 0.4 - width/2, num_hues)

for i, type_ in enumerate(hue_types):
    df_type = stats[stats['approach'] == type_]
    x_pos = [x_categories.index(cat) + offsets[i] for cat in df_type['experiment']]
    for xi, m, s in zip(x_pos, df_type['mean'], df_type['sem']):
        ax.text(xi, m + 2*s, f"{m:.2g}", ha='center', va='bottom', fontweight='bold')


plt.title(f"")
plt.xlabel("")
plt.ylabel(f"{'Subprogram' if 'synthesizer' in metric else 'Decomposition'} accuracy [%]")
fig.gca().set_ylim(0, 102)
ax.legend().set_title('')
fig.tight_layout()
# fig.savefig('../plots/svg/decomp_acc_dc.svg', transparent=True)

# ax.legend(loc='upper right', bbox_to_anchor=(0.91, 1.0))

legend = ax.get_legend()
if legend is not None and not hasattr(legend, "_ncol"):
    legend._ncol = legend._ncols if hasattr(legend, "_ncols") else 1

stats


# Quantitative Performance

In [ ]:
dataset = 'lambdabeam'
path = f"{EVAL_DIR}/{dataset}_e2e_predict_1/"
bs = 10

accuracies = load_accuracy_data(dataset, path, beam_size=bs)

for k in accuracies.keys():
    print(f"{k}: {accuracies[k]}")

# Transform data into a DataFrame suitable for seaborn plotting
df_list = []
for hue_key, subdict in accuracies.items():
    for category, values in subdict.items():
        for i, value in enumerate(values):
            df_list.append({'Category': EXP_LBL_MAP[category], 'Value': value, 'Type': hue_key, 'Seed': (i+1) * 10})

df = pd.DataFrame(df_list)
df = df[~df['Type'].isin(['Exedec', 'GT Oracle'])]  # 'SAD Oracle', 'Exedec Oracle', 'GT Oracle'

# Compute mean and std for all Category-Type combinations (needed for error bars)
# Compute mean and standard error (SEM)
stats = df.groupby(['Category', 'Type'])['Value'].agg(['mean', 'std', 'count']).reset_index()
stats['sem'] = stats['std'] / np.sqrt(stats['count'])


fig, ax = plt.subplots()
sns.barplot(data=df, x='Category', y='Value', hue='Type', errorbar=('ci', 95), ax=ax, err_kws={'color': 'gray'}, palette=PALETTE, hue_order=HUE_ORDER[:-2])

# Compute pairwise comparisons between all approaches per category
hue_types = df['Type'].unique()
x_categories = df['Category'].unique()

# Create list of all pairs for comparisons
comparisons = []
for cat in x_categories:
    cat_pairs = [(h1, h2) for i, h1 in enumerate(hue_types) for j, h2 in enumerate(hue_types) if i < j]
    for pair in cat_pairs:
        comparisons.append((pair, cat))

# Restructure comparisons for statannotations
formatted_comparisons = []
for (h1, h2), cat in comparisons:
    formatted_comparisons.append(((cat, h1), (cat, h2)))

# Create annotator
"""annotator = Annotator(ax, formatted_comparisons, data=df, x='Category', y='Value', hue='Type', order=list(x_categories))
annotator.pvalue_format = {'fontsize': 14}
annotator.configure(test='t-test_paired', text_format='star', loc='outside', verbose=2)
annotator.apply_test()
annotator.annotate(
    line_offset=0.05,     
)"""
plt.gcf().set_figwidth(7)

# Add SEM error bars manually
hue_types = df['Type'].unique()
num_hues = len(hue_types)
x_categories = list(df['Category'].unique())
width = 0.8 / num_hues
offsets = np.linspace(-0.4 + width/2, 0.4 - width/2, num_hues)

for i, type_ in enumerate(hue_types):
    df_type = stats[stats['Type'] == type_]
    x_pos = [x_categories.index(cat) + offsets[i] for cat in df_type['Category']]
    for xi, m, s in zip(x_pos, df_type['mean'], df_type['sem']):
        ax.text(xi, m + 2*s, f"{m:.2g}", ha='center', va='bottom', fontweight='bold', fontsize=14)


plt.title(f"Performance across approaches")
plt.xlabel("")
plt.ylabel("Number of solved tasks [%]")
fig.gca().set_ylim(0, 119)
ax.legend().set_title('')
ax.legend(loc='upper right')
fig.tight_layout()

# ax.legend(loc='upper right', bbox_to_anchor=(0.91, 1.0))

legend = ax.get_legend()
if legend is not None and not hasattr(legend, "_ncol"):
    legend._ncol = legend._ncols if hasattr(legend, "_ncols") else 1
fig.savefig("../plots/svg/beamoracle_lb.svg", transparent=True)
stats

# Realizability study

In [ ]:
from subgoal_realizability import summarize_by_approach, print_summary

approach_dirs = {
    "SAD (ours)": f"{EVAL_DIR}/deepcoder_e2e_predict_1/grpo_none_nsatrue/end_to_end_predict-deepcoder-run-e2e_predict_1-separate",
    "Solver-agnostic baseline": f"{EVAL_DIR}/deepcoder_e2e_predict_1/supervised_none_nsatrue/end_to_end_predict-deepcoder-run-e2e_predict_1-separate",
}

for exp in EXPERIMENTS:
    print(exp)

    analysis_df, summary_df = summarize_by_approach(
        approach_to_log_dir=approach_dirs,
        dataset_type="deepcoder",
        experiment=exp,  
        max_witnesses=5,
    )

    print_summary(summary_df, classification_column="classification")
    print(summary_df)


In [ ]:
approach_dirs = {
    "SAD (ours)": f"{EVAL_DIR}/lambdabeam_e2e_predict_1/grpo_none_nsatrue/end_to_end_predict-lambdabeam-run-e2e_predict_1-separate",
    "Solver-agnostic": f"{EVAL_DIR}/lambdabeam_e2e_predict_1/supervised_none_nsatrue/end_to_end_predict-lambdabeam-run-e2e_predict_1-separate",
}

for exp in EXPERIMENTS:
    print(exp)

    analysis_df, summary_df = summarize_by_approach(
        approach_to_log_dir=approach_dirs,
        dataset_type="lambdabeam",
        experiment=exp,  
        max_witnesses=5,
    )

    print_summary(summary_df, classification_column="classification")
    print(summary_df)


## Venn diagrams

In [ ]:
dataset = 'lambdabeam'
path = f"{EVAL_DIR}/{dataset}_e2e_predict_1/"
beam_size = 10

results = {"Exedec": {s: {k: [] * 5 for k in EXPERIMENTS} for s in SEEDS}, "No-Decomposition Ablation": {s: {k: [] * 5 for k in EXPERIMENTS} for s in SEEDS}, "SAD (ours)": {s: {k: [] * 5 for k in EXPERIMENTS} for s in SEEDS}, "SAD Oracle": {s: {k: [] * 5 for k in EXPERIMENTS} for s in SEEDS}, "Exedec Oracle": {s: {k: [] * 5 for k in EXPERIMENTS} for s in SEEDS}, "GT Oracle": {s: {k: [] * 5 for k in EXPERIMENTS} for s in SEEDS}, "Solver-agnostic baseline": {s: {k: [] * 5 for k in EXPERIMENTS} for s in SEEDS}}

for training_type in ["supervised", "baseline", "grpo", "grpo_kbest", "supervised_kbest", "supervised_oracle"]:
    if training_type == 'baseline':
            prediction_type = 'baseline'
            tt = 'supervised'
    elif training_type == 'exedec':
        prediction_type = 'separate'
        tt = 'exedec'
        gr, ca, synth = False, True, ''
    else:
        prediction_type = 'separate'
        tt = training_type
    oracle_type = ''
    if '_' not in training_type:
        oracle_type = '_none'

    p = path + f"/{tt}{oracle_type}_nsa{'false' if tt == 'exedec' else 'true'}/end_to_end_predict-{dataset}-run-e2e_predict_1-"

    exp = {k: [np.nan] * 5 for k in EXPERIMENTS}
    for experiment in EXPERIMENTS:
        for j, seed in enumerate(SEEDS):
            
            try:
                with open(p + prediction_type  + f'/tb/hparams-dataset_type={dataset},prediction_type={prediction_type},experiment={experiment},beam_size={beam_size},seed={seed},greedy=True,compute_align=False,synthesizer=nsa/results-{prediction_type}.json') as f:
                    data = json.load(f)
            except FileNotFoundError:
                # print(p + prediction_type  + f'/tb/hparams-dataset_type={dataset},prediction_type={prediction_type},experiment={experiment},beam_size={beam_size},seed={seed},greedy=True,compute_align=False,synthesizer=nsa/results-{prediction_type}.json')
                continue

            try:
                for task in data:
                    if task['false_positive'] == False:
                        suc = True
                    else:
                        suc = False
                    results[PT_MAPPING[training_type]][seed][experiment].append({"success": suc, 'Task ID': task['test_example_index'], 'Task': task, 'Solution': task['solution'], 'GT': task['ground_truth']})
            except KeyError:
                continue
    
df_list = []
for pred_type, subdict in results.items():
    for s, subsubdict in subdict.items():
        for exp, values in subsubdict.items():
            for value in values:
                df_list.append({'Approach': pred_type, 'Success': value["success"], 'Experiment': EXP_LBL_MAP[exp].replace(' ', '\n'), 'Seed': s, 'Task': value['Task'], 'Task ID': value['Task ID'], 'Solution': value['Solution'], 'GT': value['GT']})

df = pd.DataFrame(df_list)
df['Approach'].unique()

In [ ]:
# 1. Extract useful fields
df["ground_truth"] = df["Task"].apply(lambda x: x.get("ground_truth"))
df["inputs"] = df["Task"].apply(lambda x: tuple(x.get("inputs")))  # make hashable

# 2. Define a task signature = (ground_truth, inputs)
df["task_signature"] = list(zip(df["ground_truth"], df["inputs"]))

# 3. Keep only tasks that exist for all Approaches
num_approaches = df["Approach"].nunique()
valid_signatures = (
    df.groupby("task_signature")["Approach"]
      .nunique()
)
valid_signatures = valid_signatures[valid_signatures == num_approaches].index

df = df[df["task_signature"].isin(valid_signatures)].copy()

# 4. Assign Task IDs (stable per unique signature)
df["Task ID"] = df.groupby("task_signature").ngroup()

# Optional: drop helper columns
# df = df.drop(columns=["ground_truth", "inputs", "task_signature"])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

approaches = ['SAD (ours)', 'GT Oracle']
seeds = SEEDS 
df_success = df[df['Success']]

# 1. Collect counts for each region across all seeds
# Regions for 2-way: Ab (Only A), aB (Only B), AB (Intersection)
stats = {'10': [], '01': [], '11': []}

sad_exclusive = {'GT match': [], 'Length diff': [], 'used_operations': {}, 'gt_operations': {}}
remainder = {'SAD (ours)': {'GT match': [], 'Length diff': [], 'used_operations': {}, 'gt_operations': {}}, 'GT Oracle': {'GT match': [], 'Length diff': [], 'used_operations': {}, 'gt_operations': {}}}

for s in seeds:
    df_seed = df_success[df_success['Seed'] == s]
    
    set_a = set(df_seed[df_seed['Approach'] == approaches[0]]['Task ID'])
    set_b = set(df_seed[df_seed['Approach'] == approaches[1]]['Task ID'])

    for ele in list(set_a - set_b):
        t = df_seed[(df_seed['Task ID'] == ele) & (df_seed['Approach'] == 'SAD (ours)')]
        t = t['Task'].values[0]
        sad_exclusive['GT match'].append(t['solution'] == t['ground_truth'])
        sad_exclusive['Length diff'].append(len([p for p in t['solution'].split(' | ') if 'INPUT' not in p]) - len([p for p in t['ground_truth'].split(' | ') if 'INPUT' not in p]))

        for sp in t['solution'].split(' | '):
            
            if 'INPUT' not in sp:
                for prim in sp.split(' '):
                    
                    if prim == '=' or 'x' in prim or (any(char.isdigit() for char in prim) and prim != 'Scanl1'):
                        continue
                    if prim in sad_exclusive['used_operations']:
                        sad_exclusive['used_operations'][prim] += 1
                    else:
                        sad_exclusive['used_operations'][prim] = 1
        for sp in t['ground_truth'].split(' | '):
            if 'INPUT' not in sp:
                for prim in sp.split(' '):
                    if prim == '=' or 'x' in prim or (any(char.isdigit() for char in prim) and prim != 'Scanl1'):
                        continue
                    if prim in sad_exclusive['gt_operations']:
                        sad_exclusive['gt_operations'][prim] += 1
                    else:
                        sad_exclusive['gt_operations'][prim] = 1
    """for app in approaches:
        for ele in list(set_b & set_a):
            t = df_seed[(df_seed['Task ID'] == ele) & (df_seed['Approach'] == app)]
            t = t['Task'].values[0]
            remainder[app]['GT match'].append(t['solution'] == t['ground_truth'])
            remainder[app]['Length diff'].append(len([p for p in t['solution'].split(' | ') if 'INPUT' not in p]) - len([p for p in t['ground_truth'].split(' | ') if 'INPUT' not in p]))
            for op in t['solution'].split(' | '):
                if 'INPUT' not in p:
                    prim = op.split(' ')[2] # first is variable xN, second is =
                    if prim in remainder[app]['used_operations']:
                        remainder[app]['used_operations'][prim] += 1
                    else:
                        remainder[app]['used_operations'][prim] = 1

            for op in t['ground_truth'].split(' | '):
                if 'INPUT' not in p:
                    prim = op.split(' ')[2]
                    if prim in remainder[app]['gt_operations']:
                        remainder[app]['gt_operations'][prim] += 1
                    else:
                        remainder[app]['gt_operations'][prim] = 1"""
    
    stats['10'].append(len(set_a - set_b))  # Only A
    stats['01'].append(len(set_b - set_a))  # Only B
    stats['11'].append(len(set_a & set_b))  # Intersection

# 2. Compute Mean and Std
results = {k: (np.mean(v), np.std(v)) for k, v in stats.items()}

# 3. Plot
plt.figure()
# subsets=(Ab, aB, AB)
v = venn2(subsets=(1, 1, 1), set_labels=approaches, set_colors=[PALETTE[approaches[0]], PALETTE[approaches[1]]]) 

# 4. Manually override the labels with our stats
# Mapping internal IDs to our binary keys
id_map = {'10': '10', '01': '01', '11': '11'}

for region_id, key in id_map.items():
    label = v.get_label_by_id(region_id)
    if label:
        mean_val, std_val = results[key]
        print(region_id, f"{mean_val:.1f}±{std_val:.1f}")
        label.set_text(f"{mean_val:.1f}\n±{std_val:.1f}")

plt.title(f"")
plt.gcf().set_figwidth(7)
plt.tight_layout()
#plt.gcf().savefig("../plots/svg/venn_dc.svg", transparent=True)


#### SAD exclusive analysis

In [ ]:
for data in [sad_exclusive]: 
    print(f"Percentage of GT matches: {np.array(data['GT match']).sum() / len(data['GT match'])}")
    print(f"Length diff to GT: Percentage longer {len(np.array(data['Length diff'])[np.array(data['Length diff']) > 0]) / len(data['Length diff'])}; Percentage shorter: {len(np.array(data['Length diff'])[np.array(data['Length diff']) < 0]) / len(data['Length diff'])}; Percentage equal: {len(np.array(data['Length diff'])[np.array(data['Length diff']) == 0]) / len(data['Length diff'])}")

    df1 = pd.DataFrame(list(data['used_operations'].items()), columns=['Operation', 'Count'])
    df1['Approach'] = 'SAD (ours)'
    df1['Count'] = df1['Count']

    df2 = pd.DataFrame(list(data['gt_operations'].items()), columns=['Operation', 'Count'])
    df2['Approach'] = 'GT Oracle'
    df2['Count'] = df2['Count']

    # 3. Combine them into one "Long-Format" DataFrame
    df_plot = pd.concat([df1, df2])


    # 4. Plot
    plt.figure()
    sns.set_theme(style="whitegrid")
    plt.gcf().set_figwidth(7)

    sns.barplot(data=df_plot, x='Operation', y='Count', hue='Approach', palette=PALETTE)

    plt.xticks(rotation=45)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.gcf().savefig("../plots/svg/dslusage_lb.svg", transparent=True)

#### Comparison Between SAD solutions and GT oracle solutions

In [ ]:
for data in [remainder['SAD (ours)'], remainder['GT Oracle']]:
    print(f"Percentage of GT matches: {np.array(data['GT match']).sum() / len(data['GT match'])}")
    print(f"Length diff to GT: Percentage longer {len(np.array(data['Length diff'])[np.array(data['Length diff']) > 0]) / len(data['Length diff'])}; Percentage shorter: {len(np.array(data['Length diff'])[np.array(data['Length diff']) < 0]) / len(data['Length diff'])}; Percentage equal: {len(np.array(data['Length diff'])[np.array(data['Length diff']) == 0]) / len(data['Length diff'])}")

    df1 = pd.DataFrame(list(data['used_operations'].items()), columns=['Operation', 'Count'])
    df1['Approach'] = 'SAD exclusive'
    df1['Count'] = df1['Count']

    df2 = pd.DataFrame(list(data['gt_operations'].items()), columns=['Operation', 'Count'])
    df2['Approach'] = 'GT'
    df2['Count'] = df2['Count']

    # 3. Combine them into one "Long-Format" DataFrame
    df = pd.concat([df1, df2])


    # 4. Plot
    plt.figure(figsize=(10, 6))
    sns.set_theme(style="whitegrid")

    sns.barplot(data=df, x='Operation', y='Count', hue='Approach', palette='magma')

    plt.title('Operations used by Model but missing/different in GT', fontsize=14)
    plt.xticks(rotation=45)
    plt.ylabel('Difference in Count')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_venn import venn3
from collections import defaultdict

approaches = ['SAD (ours)', 'GT Oracle', 'Solver-agnostic baseline']

# 1. Define the 3-bit binary regions used by venn3
# These IDs correspond to: Only A, Only B, AB, Only C, AC, BC, ABC
regions = ['100', '010', '110', '001', '101', '011', '111']

# 2. Calculate counts per seed
seed_counts = defaultdict(list)
seeds = SEEDS # [10, 20, 30] # ToDo: seeds

for s in seeds:
    df_seed = df_success[df_success['Seed'] == s]
    
    # Create sets for this seed
    sets = {
        'A': set(df_seed[df_seed['Approach'] == approaches[0]]['Task ID']),
        'B': set(df_seed[df_seed['Approach'] == approaches[1]]['Task ID']),
        'C': set(df_seed[df_seed['Approach'] == approaches[2]]['Task ID'])
    }
    
    # Calculate what falls into each specific Venn area
    # Note: '110' means (A & B) - C
    venn_data = {
        '100': len(sets['A'] - sets['B'] - sets['C']),
        '010': len(sets['B'] - sets['A'] - sets['C']),
        '110': len((sets['A'] & sets['B']) - sets['C']),
        '001': len(sets['C'] - sets['A'] - sets['B']),
        '101': len((sets['A'] & sets['C']) - sets['B']),
        '011': len((sets['B'] & sets['C']) - sets['A']),
        '111': len(sets['A'] & sets['B'] & sets['C'])
    }
    
    for r in regions:
        seed_counts[r].append(venn_data[r])

# 3. Plotting
plt.figure(figsize=(10, 8))

# We initialize with ones just to create the shapes
v = venn3(subsets=(1, 1, 1, 1, 1, 1, 1), set_labels=approaches)

# 4. Manually overwrite the labels with Mean ± Std
for r in regions:
    mean = np.mean(seed_counts[r])
    std = np.std(seed_counts[r])
    
    lbl = v.get_label_by_id(r)
    if lbl:
        # Format: "12.3\n±1.2"
        print(r, f"{mean:.1f}±{std:.1f}")
        lbl.set_text(f"{mean:.1f}\n±{std:.1f}")

plt.title(f"Mean Solved Tasks across Seeds ({dataset.title()})")
plt.show()

# By length

In [ ]:
dataset = 'lambdabeam'
path = f"{EVAL_DIR}/{dataset}_e2e_predict_1/"
bs = 10

accuracies = load_accuracy_by_length(dataset, path, beam_size=bs)

df_sad = pd.DataFrame(accuracies['SAD (ours)'])
df_sad['Approach'] = 'SAD (ours)'

df_baseline = pd.DataFrame(accuracies['Solver-blind baseline'])
df_baseline['Approach'] = 'Solver-blind baseline'

"""df_exedec = pd.DataFrame(accuracies['Exedec'] * 100)
df_exedec['Approach'] = 'Exedec'"""

# 3. Combine the datasets
df = pd.concat([df_sad, df_baseline], ignore_index=True)

df_plot = df
# 4. Convert 'success' boolean to numeric (1.0 or 0.0) for mean calculation
df_plot['success'] = df_plot['success'].astype(float) * 100

# 5. Create the plot
fig = plt.figure()
sns.set_style("whitegrid")

sns.pointplot(
    data=df_plot, 
    x='length', 
    y='success', 
    hue='Approach', 
    markers='o', 
    palette=PALETTE,
    dodge=0.05,           
    linestyles='-',      
    native_scale=True    
)

# Adjust line width globally for the current axes
plt.setp(plt.gca().get_lines(), linewidth=3)

# Formatting
#plt.title(f'Performance by number of GT decompositions')
plt.xlabel('Program length')
plt.ylabel('Number of solved tasks [%]')
plt.ylim(0, 105)  
plt.gcf().set_figwidth(7)
plt.legend(title='')
fig.gca().set_xticks(np.arange(1, 6, 1))
fig.tight_layout()
plt.gcf().savefig("../plots/svg/perflen_lb.svg", transparent=True)


# DSL operation usage between solutions and GT

In [ ]:
dataset = 'deepcoder'
path = f"{EVAL_DIR}/{dataset}_e2e_predict_1/"
bs = 10

accuracies = load_solutions_and_gt(dataset, path, beam_size=bs)

df_sad = pd.DataFrame(accuracies['SAD (ours)'])
df_sad['Approach'] = 'SAD (ours)'

df_baseline = pd.DataFrame(accuracies['Solver-blind baseline'])
df_baseline['Approach'] = 'Solver-blind baseline'

"""df_exedec = pd.DataFrame(accuracies['Exedec'])
df_exedec['Approach'] = 'Exedec'"""

# 3. Combine the datasets
df = pd.concat([df_sad, df_baseline], ignore_index=True)
print(df)

df['success'] = df['success'].astype(bool)
    
# 2. Identify the size of each task group
# Using 'gt' as the key is safer if task_idx isn't unique across categories
group_key = ['seed', 'category', 'task_idx']
df['group_size'] = df.groupby(group_key)['Approach'].transform('count')

# 3. Filter for Group 3 (All 3 approaches present)
# AND ensure every approach in that group succeeded
all_solved = df.groupby(group_key)['success'].transform('all')

# We filter for group_size == 3 to ensure a fair 3-way comparison
df = df[(df['group_size'] == 2) & all_solved].copy()

print(f"Tasks in common (Group 3): {len(df[df['group_size'] == 2])}")
print(f"Tasks in common where all 3 succeeded: {len(df)}")


def extract_ops(text):
    if not isinstance(text, str): return []
    # Matches capitalized words (DSL ops), excluding 'INPUT'
    ops = re.findall(r'([A-Z][a-zA-Z0-9]+)(?=\s|\()', text)
    return [op for op in ops if op != 'INPUT']

# 1. Extract operations
df['gt_ops'] = df['gt'].apply(extract_ops)
df['sol_ops'] = df['solution'].apply(extract_ops)

# 2. Count occurrences per (Approach, Seed, Operation)
# Ground Truth counts
gt_counts = (df.explode('gt_ops')
                .groupby(['Approach', 'seed', 'gt_ops'])
                .size()
                .reset_index(name='gt_count')
                .rename(columns={'gt_ops': 'Operation'}))

# Solution counts
sol_counts = (df.explode('sol_ops')
                .groupby(['Approach', 'seed', 'sol_ops'])
                .size()
                .reset_index(name='sol_count')
                .rename(columns={'sol_ops': 'Operation'}))

# 3. Merge counts and handle missing values (zeros)
# We use an outer join to ensure we capture ops that might only appear in GT or only in Solution
merged = pd.merge(
    gt_counts, 
    sol_counts, 
    on=['Approach', 'seed', 'Operation'], 
    how='outer'
).fillna(0)

# 4. Calculate the Difference per seed
# Positive = Solution uses it more | Negative = GT uses it more
merged['diff'] = merged['sol_count'] - merged['gt_count']

# 5. Plotting
plt.figure(figsize=(12, 10))
sns.set_theme(style="whitegrid")

# Determine plot order based on the magnitude of the mean difference
avg_diffs = merged.groupby('Operation')['diff'].apply(lambda x: x.abs().mean()).sort_values(ascending=False)
plot_order = avg_diffs.index

# Use errorbar='sd' to show standard deviation across seeds
ax = sns.barplot(
    data=merged, 
    y='Operation', 
    x='diff', 
    hue='Approach', 
    order=plot_order,
    capsize=.2,
    palette=PALETTE
)

# Styling
plt.axvline(0, color='black', linewidth=1.5, linestyle='--')
plt.title('Mean Delta Frequency of DSL Operations (Solution - GT) with SD', fontsize=14)
plt.xlabel('Frequency Difference (Average across seeds)')
plt.ylabel('DSL Operation')
plt.tight_layout()




## Training curves

In [ ]:
paths = {'SAD (ours)': ["grpo/tb/adr=0.1,ara=True,dr=0.1,e=NONE,ed=512,hd=1024,l=0.0001,md=60,mpced=180,npb=32,s=20,scnpr=0.0,ura=True,dm=coupled",
        "grpo/tb/adr=0.1,ara=True,dr=0.1,e=NONE,ed=512,hd=1024,l=0.0001,md=60,mpced=180,npb=32,s=10,scnpr=0.0,ura=True,dm=coupled"],
        '$\mathcal{L}_{\\text{sup}}$ ablation': ["grpo_nosup/tb/adr=0.1,ara=True,dr=0.1,e=NONE,ed=512,hd=1024,l=0.0001,md=60,mpced=180,npb=32,s=10,scnpr=0.0,ura=True,dm=coupled", 
        "grpo_nosup/tb/adr=0.1,ara=True,dr=0.1,e=NONE,ed=512,hd=1024,l=0.0001,md=60,mpced=180,npb=32,s=10,scnpr=0.0,ura=True,dm=coupled"]
    }
data_list = []
key = 'train/greedy accuracy' # 'train/accuracy' 

for t_type in ['SAD (ours)', '$\mathcal{L}_{\\text{sup}}$ ablation']:
    for i, dom in enumerate(['deepcoder', 'lambdabeam']):
        p = f'{RESULTS_DIR}/exedec_train_{dom}_run-1/spec_decomposer_model/' + paths[t_type][i]
        ea = event_accumulator.EventAccumulator(p)
        ea.Reload()
        try:
            tensor_events = ea.Tensors(key)
            for event in tensor_events:
                # Convert the proto to a numpy object
                # .item() extracts the single float from the 0-d array
                value = tf.make_ndarray(event.tensor_proto).item()
                
                data_list.append({
                    'Step': event.step,
                    'Accuracy': value * 100,
                    'Supervision': t_type,
                    'Domain': dom.title()
                })
        except KeyError:
            print(f"Warning: Tag '{key}' not found in {p}")

df = pd.DataFrame(data_list)

# --- Plotting with Seaborn ---
fig = plt.figure()
sns.set_theme(style="whitegrid")

sns.lineplot(
    data=df, 
    x='Step', 
    y='Accuracy', 
    hue='Supervision', 
    style='Domain',
    linewidth=2.5,
    palette=PALETTE
)

plt.title('')
plt.ylabel('Training synthesis accuracy [%]')
plt.xlabel('Training steps')

plt.gcf().set_figwidth(7)
fig.gca().set_xticks(np.arange(100000, 600000, 100000), ['100k', '200k', '300k', '400k', '500k'])
#fig.gca().get_legend().remove()
plt.tight_layout()
plt.ylim(0, 105)
fig.savefig("../plots/svg/lsup_synthacc.svg", transparent=True)

## Illustrative examples
#### Procedural mismatch

In [ ]:
# Path to your JSON file
ds = 'lambdabeam'
s = 40
exp = 'NONE' # 'LENGTH_GENERALIZATION'
file_path = f"{EVAL_DIR}/{ds}_e2e_predict_1/grpo_none_nsatrue/end_to_end_predict-{ds}-run-e2e_predict_1-separate/tb/hparams-dataset_type={ds},prediction_type=separate,experiment={exp},beam_size=10,seed={s},greedy=True,compute_align=False,synthesizer=nsa/results-separate.json" 

# Load JSON
with open(file_path, "r") as f:
    data = json.load(f)

# Convert to DataFrame
df_sad = pd.DataFrame(data)


file_path = f"{EVAL_DIR}/{ds}_e2e_predict_1/supervised_none_nsatrue/end_to_end_predict-{ds}-run-e2e_predict_1-separate/tb/hparams-dataset_type={ds},prediction_type=separate,experiment={exp},beam_size=10,seed={s},greedy=True,compute_align=False,synthesizer=nsa/results-separate.json" 

# Load JSON
with open(file_path, "r") as f:
    data = json.load(f)

# Convert to DataFrame
df_sup = pd.DataFrame(data)

merged = df_sad.merge(
    df_sup[["test_example_index", "false_positive", "solution", "subgoals"]],
    on="test_example_index",
    how="left",
    suffixes=("_sad", "_sup")
)

In [ ]:
df_suc = merged[(merged["false_positive_sad"] == False) & (merged["false_positive_sup"] != False)]
c = 0
for _, row in df_suc.iterrows():
    if 'If' in row['ground_truth']: # len([ele for ele in row['ground_truth'].split(' | ') if 'INPUT' not in ele]) > len([ele for ele in row['solution_sad'].split(' | ') if 'INPUT' not in ele]): # row['ground_truth'] == row['solution_sup'] and 
        print(row['solution_sad'])
        print(row['solution_sup'])
        print(row['ground_truth'])
        print(row['subgoals_sad'])
        print(row['inputs'])
        print(row)
        c += 1
        print('_' * 80)
        if 10 <= c <= 20:
            i = 1/0

#### Structural mismatch

In [ ]:
# Tasks solved in A but not in B
result = merged[(merged["false_positive_sad"] == False) & (merged["false_positive_sup"] != False)]


c = 0
for _, row in result.iterrows():
    if row['subgoals_sad'][0] != row['outputs']: # 'x0 = [ -9 -49 ]' not in row['inputs'] and row['solution_sad'] == row['ground_truth']:
        print(row['subgoals_sup'])
        print(row['subgoals_sad'])
        print(row['solution_sad'])
        print(row['ground_truth'])
        print(row['inputs'])
        print(row['subgoals_sad'][0] != row['outputs'])
        print(row)

        c += 1
        print('_' * 80)
        if c >= 50:
            i = 1/0